# 01 — Preprocess

動画または多視点画像から、COLMAPへ渡す多視点画像を準備します。動画は一定間隔でフレームへ変換し、多視点画像はそのまま使用します。最後に、COLMAP入力画像の品質を確認します。

## 1. 環境確認

必要なツールとGPUを確認します。ffmpegは動画のフレーム抽出、COLMAPはSfM、PyTorchとgsplatはGaussianの訓練に使います。

`Environment ready`と表示されれば、実行環境は整っています。

In [ ]:
from gs_tutorial.environment import print_environment

environment_ready = print_environment()
print(f"Environment ready: {environment_ready}")

## 2. 設定と入力データ

`fps`は動画から取り出すフレーム数、`max_width`はCOLMAPへ渡す画像の最大幅です。高解像度画像や多くのフレームは細部を残せますが、処理時間も増えます。動画には`sequential`、順不同の多視点画像には`exhaustive` matcherを使います。

実行結果で入力の種類、path、出力先、matcherを確認します。

In [ ]:
from pathlib import Path

from IPython.display import Video, display

from gs_tutorial.config import load_config

cwd = Path.cwd().resolve()
repo = cwd.parent if (cwd.parent / "pyproject.toml").is_file() else cwd
config_path = repo / "configs/video.yaml"  # Use images.yaml for multi-view image input
cfg = load_config(config_path)
source = repo / cfg.input.source
root = repo / cfg.project_dir
print("Config path:", config_path)
print("Input path:", f"({cfg.input.kind}) {source}")
print("Output directory:", root)
print("COLMAP matcher:", cfg.colmap.matcher)
if not source.exists():
    raise FileNotFoundError(source)

In [ ]:
if cfg.input.kind == "video":
    display(Video(str(source), embed=False, width=720))
else:
    input_images = sorted(
        path
        for path in source.iterdir()
        if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".heic", ".heif", ".webp"}
    )
    print(f"{len(input_images)} input images")
    display(input_images[:5])

## 3. 多視点画像の作成と品質確認

動画はフレーム抽出し、多視点画像に変換します。入力が既に多視点画像の場合は変換をスキップします。その後は共通して、SfMに使う画像を選びます。

- **Laplacian分散**: 値が小さい画像をぼけとして除外します。
- **Difference hash**: 直前の採用画像に似すぎた画像を除外します。順不同の画像では無効化できます。

グラフでは緑が採用、赤が除外、破線がぼけ閾値です。採用画像が少なすぎないか確認します。

In [ ]:
import json

from gs_tutorial.preprocessing import extract_video_frames, reports_as_dicts, select_images

raw_dir, image_dir = root / "raw_frames", root / "images"
report_path = root / "frame_report.json"
root.mkdir(parents=True, exist_ok=True)
if cfg.input.kind == "video":
    frames = sorted(raw_dir.glob("frame_*.jpg"))
    if frames:
        print(f"Video frames already extracted: reusing {len(frames)} images")
    else:
        frames = extract_video_frames(
            source, raw_dir, fps=cfg.input.fps, max_width=cfg.input.max_width
        )
    selection_source = raw_dir
else:
    selection_source = source
selected = sorted(image_dir.glob("image_*.jpg"))
reports = []
if selected:
    print(f"Image selection already complete: reusing {len(selected)} images")
else:
    reports = select_images(
        selection_source,
        image_dir,
        max_width=cfg.input.max_width,
        blur_threshold=cfg.input.blur_threshold,
        duplicate_hamming_threshold=cfg.input.duplicate_hamming_threshold,
    )
    report_path.write_text(
        json.dumps(reports_as_dicts(reports), indent=2, ensure_ascii=False), encoding="utf-8"
    )
    selected = sorted(image_dir.glob("image_*.jpg"))
    print(f"Image selection complete: kept {len(selected)} / {len(reports)} images")

In [ ]:
from collections import Counter
from typing import cast

import matplotlib.pyplot as plt

if reports:
    report_data = reports_as_dicts(reports)
elif report_path.is_file():
    report_data = json.loads(report_path.read_text(encoding="utf-8"))
else:
    report_data = []
if report_data:
    print("Selection summary:", dict(Counter(item["reason"] for item in report_data)))
    scores = [cast(float, item["sharpness"]) for item in report_data]
    colors = ["#2ca02c" if item["kept"] else "#d62728" for item in report_data]
    figure, axis = plt.subplots(figsize=(12, 4))
    axis.scatter(range(len(scores)), scores, c=colors, s=18)
    axis.axhline(cfg.input.blur_threshold, color="black", linestyle="--", label="blur threshold")
    axis.set(xlabel="Input image index", ylabel="Laplacian variance", title="Image sharpness")
    axis.legend()
    figure.tight_layout()
else:
    print(
        "Reusing existing images; skipping the quality graph because frame_report.json is missing."
    )

## 4. COLMAP入力の目視確認

サムネイルを見て、被写体を複数方向から撮れているか、近い視点同士が十分に重なるか、大きなぼけが発生していないかを確認します。

In [ ]:
from PIL import Image

selected = sorted(image_dir.glob("image_*.jpg"))
if len(selected) < 3:
    raise RuntimeError(
        f"At least 3 images are required for COLMAP, but only {len(selected)} were selected."
    )
sample = selected[:: max(1, len(selected) // 12)][:12]
figure, axes = plt.subplots(3, 4, figsize=(14, 9))
for axis, path in zip(axes.flat, sample):
    with Image.open(path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(path.name)
    axis.axis("off")
for axis in axes.flat[len(sample) :]:
    axis.axis("off")
figure.suptitle(f"COLMAP input: {len(selected)} images")
figure.tight_layout()